This notebook runs scripts to find out which BigQuery features the free sandbox allows, because they decide how raw tables are loaded and how dbt models are built for this project.
This notebook writes to `<gcp-project>.test_dataset` which is a throwaway dataset for testing purposes only.

In [2]:
import datetime
import os

import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

In [3]:
load_dotenv()
GCP_PROJECT = os.getenv("GCP_PROJECT")
if not GCP_PROJECT:
    raise RuntimeError("GCP_PROJECT is not set. Check .env file.")
TEST_DATASET = "test_dataset"
test_tbl_id = f"{GCP_PROJECT}.{TEST_DATASET}.test_table"
partitioned_test_tbl_id = f"{GCP_PROJECT}.{TEST_DATASET}.partitioned_test_table"

In [4]:
client = bigquery.Client(project=GCP_PROJECT)
client.create_dataset(bigquery.Dataset(f"{GCP_PROJECT}.{TEST_DATASET}"), exists_ok=True)
print("")

## 1. Load job from a DataFrame

In [5]:
df = pd.DataFrame({"id": [1, 2, 3], "name": ["Alice", "Bob", "Charlie"], "age": [25, 30, 35]})

job = client.load_table_from_dataframe(
    df,
    test_tbl_id,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),  # also: WRITE_APPEND, WRITE_EMPTY
)
job.result()
print(f"Loaded {job.output_rows} rows into {TEST_DATASET}.test_table")

Loaded 3 rows into test_dataset.test_table


## 2. Partitioned table with old dates


In [7]:
# Six rows are loaded into a table partitioned by `date_column`. Only one date (2026-10-02) is within the last 60 days.
df_with_date = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "name": ["Alice", "Bob", "Charlie", "David", "Eve", "Frank"],
    "age": [25, 30, 35, 40, 45, 50],
    "date_column": [
        datetime.date(2023, 1, 1), datetime.date(2023, 1, 2), datetime.date(2024, 1, 1),
        datetime.date(2024, 1, 2), datetime.date(2025, 1, 1), datetime.date(2026, 10, 2),
    ],
})

partitioned_job = client.load_table_from_dataframe(
    df_with_date,
    partitioned_test_tbl_id,
    job_config=bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",
        time_partitioning=bigquery.TimePartitioning(
            type_=bigquery.TimePartitioningType.DAY,
            field="date_column",
            expiration_ms=None,  # ask for no partition expiry
        ),
    ),
)
partitioned_job.result()
print(f"Loaded {partitioned_job.output_rows} rows.")

Loaded 6 rows.


In [8]:
# how many of the loaded rows are still there?
surviving = client.query(f"SELECT count(*) AS n FROM `{partitioned_test_tbl_id}`").to_dataframe().iloc[0, 0]
print(f"rows loaded: {len(df_with_date)}, rows surviving: {surviving}")

rows loaded: 6, rows surviving: 1


# 3. Expiration settings

In [9]:
tbl = client.get_table(partitioned_test_tbl_id)
print(f"table expiry before: {tbl.expires}   now: {datetime.datetime.now(datetime.timezone.utc)}")

tbl.expires = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=365)
tbl = client.update_table(tbl, ["expires"])
print(f"table expiry after asking for +365 days: {tbl.expires}")

table expiry before: 2027-10-20 06:02:44.328000+00:00   now: 2026-10-20 12:48:18.647189+00:00
table expiry after asking for +365 days: 2027-10-20 12:48:18.647000+00:00


In [10]:
tbl = client.get_table(partitioned_test_tbl_id)
print("Table-level expiration (the whole table is deleted):", tbl.expires)
if tbl.time_partitioning:
    print("Partition field:", tbl.time_partitioning.field)
    print("Partition expiration ms:", tbl.time_partitioning.expiration_ms)

dataset = client.get_dataset(tbl.dataset_id)
print("Dataset default partition expiration ms:", dataset.default_partition_expiration_ms)
print("Dataset default table expiration ms:", dataset.default_table_expiration_ms)

Table-level expiration (the whole table is deleted): 2027-10-20 12:48:18.647000+00:00
Partition field: date_column
Partition expiration ms: 5184000000
Dataset default partition expiration ms: 5184000000
Dataset default table expiration ms: 5184000000


The dataset defaults for table and partition expiration are both 5,184,000,000 ms, which is 60 days. Setting the table-level expiry to one year was accepted, but the partition expiration stayed at 60 days, so old partitions still disappear.

We'll treat every table as perishable and create a DuckDB raw dataset restoring bq datasets.

# 4. DML and streaming inserts

In [22]:
# might not work in sandbox mode
try:
    client.query(f"INSERT INTO `{test_tbl_id}` (id, name, age) VALUES (4, 'Dan', 40)").result()
    print("DML INSERT: allowed")
except Exception as e:
    print(f"DML INSERT refused")

try:
    errors = client.insert_rows_json(test_tbl_id, [{"id": 4, "name": "Dan", "age": 40}])
    print("Streaming insert returned:", errors if errors else "no errors (allowed)")
except Exception as e:
    print(f"Streaming insert refused")

DML INSERT refused
Streaming insert refused


dbt's incremental strategies on BigQuery rely on DML (`MERGE` or `INSERT`), if we're in sandbox mode of bq, we are limited to using views and tables that are fully rebuilt on each run.

Clean up
Set `CLEAN_UP = True` to remove the test tables created above.

In [21]:
CLEAN_UP = True
if CLEAN_UP:
    for table_id in (test_tbl_id, partitioned_test_tbl_id):
        client.delete_table(table_id, not_found_ok=True)
        print("deleted", table_id.split(".")[1:])
else:
    print("Left in place. Set CLEAN_UP = True to remove the test tables.")

deleted ['test_dataset', 'test_table']
deleted ['test_dataset', 'partitioned_test_table']
